In [1]:
# notebook: 09_figures_5_1_5_2.ipynb
# ============================================================================
# CMVTS Extension — Figures for Section 5.1 (penetration spectrum) & 5.2
#                    (predictor–outcome validation), house style
# ----------------------------------------------------------------------------
# Same figure spec as notebook 08:
#   seaborn greyscale, NO caption in image, dpi=600, save BOTH png+pdf,
#   legend (if any) at the BOTTOM. Reuses the identical style block so all
#   manuscript figures are visually consistent.
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = "."
FIG_DPI = 600

# ----------------------------------------------------------------------------
# Data (from confirmed notebooks 02, 03, 07)
# ----------------------------------------------------------------------------
# 5.1 — target-country proxy penetration spectrum (Findex 2024, weighted %)
PEN = pd.DataFrame({
    "Account":            [91.8,70.6,60.0,56.3,50.2,43.3,39.0,37.7,27.3],
    "Card use (fin8)":    [46.8,58.4,36.8,27.4,11.8, 9.1,25.4,26.5, 8.4],
    "Digital merchant pay":[51.3,51.1, 9.3,13.6,13.4, 2.8,19.8,21.5, 4.3],
}, index=["Thailand","Viet Nam","Nepal","Indonesia","Philippines",
          "Bangladesh","Cambodia","Lao PDR","Pakistan"])

# 5.2 — predictor (macro-CMVTS equal weight) vs outcome (realized divergence)
MACRO_CMVTS = {
    "Indonesia":0.6734,"Thailand":0.7762,"Viet Nam":0.6842,"Philippines":0.5650,
    "Bangladesh":0.5561,"Cambodia":0.6034,"Nepal":0.5874,"Pakistan":0.4412,"Lao PDR":0.5434,
}
Y_DIVERGENCE = {
    "Indonesia":0.0708,"Thailand":0.0092,"Viet Nam":0.0000,"Philippines":0.1809,
    "Bangladesh":0.2110,"Cambodia":0.0810,"Nepal":0.0332,"Pakistan":0.2192,"Lao PDR":0.0753,
}

# ----------------------------------------------------------------------------
# Shared house style (identical to notebook 08)
# ----------------------------------------------------------------------------
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size":11, "axes.edgecolor":"0.2", "axes.linewidth":0.8,
    "grid.color":"0.85", "figure.dpi":120,
})

def save(fig, name):
    for ext in ("png","pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png / {name}.pdf")

# ============================================================================
# FIGURE 5.1 — grouped horizontal bars, greyscale by indicator, legend bottom
# ============================================================================
# order countries by account penetration (descending) for a clean gradient
order = PEN["Account"].sort_values(ascending=True).index      # ascending -> top=highest in barh
P = PEN.loc[order]
GREYS = ["0.20","0.50","0.75"]                                # 3 indicators
ind_names = list(P.columns)

fig, ax = plt.subplots(figsize=(6.8,5.2))
n_ind = len(ind_names)
ypos = np.arange(len(P))
h = 0.8 / n_ind
for i, (col, g) in enumerate(zip(ind_names, GREYS)):
    ax.barh(ypos + (i - (n_ind-1)/2)*h, P[col], height=h,
            color=g, edgecolor="black", linewidth=0.6, label=col, zorder=3)
ax.set_yticks(ypos); ax.set_yticklabels(P.index)
ax.set_xlabel("Weighted penetration (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5,-0.10),
          ncol=3, frameon=False, handletextpad=0.5, columnspacing=1.5)
fig.tight_layout()
save(fig, "fig_5_1_penetration_spectrum")

# ============================================================================
# FIGURE 5.2 — predictor vs outcome scatter + regression, greyscale
# ============================================================================
x = pd.Series(MACRO_CMVTS); y = pd.Series(Y_DIVERGENCE)
common = x.index
fig, ax = plt.subplots(figsize=(6.4,5.2))
ax.scatter(x, y, s=90, c="0.35", edgecolors="black", linewidths=0.8, zorder=3)
# regression line
b, a = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 50)
ax.plot(xs, a + b*xs, color="0.1", lw=1.3, zorder=2)
# country labels
for c in common:
    ax.annotate(c, (x[c], y[c]), xytext=(4,4), textcoords="offset points",
                fontsize=8.5, color="0.1")
rs, ps = stats.spearmanr(x, y)
rp, pp = stats.pearsonr(x, y)
# stats box (in-axes, not a caption)
ax.text(0.03, 0.04,
        f"Spearman $\\rho$ = {rs:+.2f} (p = {ps:.3f})\nPearson r = {rp:+.2f} (p = {pp:.3f})",
        transform=ax.transAxes, fontsize=9, va="bottom", ha="left",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.5", lw=0.7))
ax.set_xlabel("Transferability  (macro-CMVTS)")
ax.set_ylabel("Realized behavioural divergence  (JSD)")
fig.tight_layout()
save(fig, "fig_5_2_predictor_outcome")

# ============================================================================
# FIGURE 5.2b (optional companion) — outcome vs one raw driver for intuition
#   shows realized divergence against target card-activity penetration, the
#   quantity the outcome summarizes. Greyscale, consistent style.
# ============================================================================
CARD_ACTIVE = {  # fin8 weighted % from notebook 02
    "Indonesia":27.4,"Thailand":46.8,"Viet Nam":58.4,"Philippines":11.8,
    "Bangladesh":9.1,"Cambodia":25.4,"Nepal":36.8,"Pakistan":8.4,"Lao PDR":26.5,
}
cx = pd.Series(CARD_ACTIVE); cy = pd.Series(Y_DIVERGENCE)
fig, ax = plt.subplots(figsize=(6.4,5.0))
ax.scatter(cx, cy, s=90, c="0.35", edgecolors="black", linewidths=0.8, zorder=3)
b,a=np.polyfit(cx,cy,1); xs=np.linspace(cx.min(),cx.max(),50)
ax.plot(xs,a+b*xs,color="0.1",lw=1.3,zorder=2)
for c in cx.index:
    ax.annotate(c,(cx[c],cy[c]),xytext=(4,4),textcoords="offset points",
                fontsize=8.5,color="0.1")
ax.set_xlabel("Target card-activity penetration (%)")
ax.set_ylabel("Realized behavioural divergence  (JSD)")
fig.tight_layout()
save(fig, "fig_5_2b_outcome_vs_penetration")

print("\nAll Section 5.1/5.2 figures: greyscale, dpi=600, png+pdf, legend bottom, no captions.")

saved fig_5_1_penetration_spectrum.png / fig_5_1_penetration_spectrum.pdf
saved fig_5_2_predictor_outcome.png / fig_5_2_predictor_outcome.pdf
saved fig_5_2b_outcome_vs_penetration.png / fig_5_2b_outcome_vs_penetration.pdf

All Section 5.1/5.2 figures: greyscale, dpi=600, png+pdf, legend bottom, no captions.
